# 02 - Preprocessing: Construcao da camada silver em Spark SQL, com filtros explicitos, feature engineering e materializacao em Parquet particionado.

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/work/notebooks')

from _lib import build_spark, DATA_GLOB, LOOKUP_PATH, SILVER_PATH, BOUNDS_PATH

spark = build_spark('nyc-rideshare-preprocessing')

In [2]:
spark.read.parquet(DATA_GLOB).createOrReplaceTempView('trips_raw')
(spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(LOOKUP_PATH)
    .createOrReplaceTempView('taxi_zone_lookup'))

# Recorte temporal: Mar-Ago 2023 (6 meses). O arquivo cobre Dez/2021-Ago/2023.
# Filtro aqui evita OOM nos workers (4G cada) e garante que o SPLIT_DATE=2023-06-01
# tenha 3 meses de treino e 3 de teste conforme CLAUDE.md.
spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW trips_bronze AS
    SELECT * FROM trips_raw
    WHERE date >= '2023-03-01'
      AND date <  '2023-09-01'
""")

spark.sql('SELECT COUNT(*) AS bronze_rows FROM trips_bronze').show()


+-----------+
|bronze_rows|
+-----------+
|  116225448|
+-----------+



In [3]:
# Bounds por percentil 0.1% e 99.9% para filtros da silver.
# speed = trip_length / (total_ride_time / 3600)  [mph]
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW cleaning_bounds AS
WITH fare_bounds AS (
    SELECT
        PERCENTILE_APPROX(passenger_fare, 0.001) AS fare_low,
        PERCENTILE_APPROX(passenger_fare, 0.999) AS fare_high
    FROM trips_bronze
    WHERE passenger_fare > 0
),
speed_bounds AS (
    SELECT
        PERCENTILE_APPROX(trip_length / (total_ride_time / 3600.0), 0.001) AS speed_low,
        PERCENTILE_APPROX(trip_length / (total_ride_time / 3600.0), 0.999) AS speed_high
    FROM trips_bronze
    WHERE trip_length > 0
      AND total_ride_time > 60
)
SELECT * FROM fare_bounds CROSS JOIN speed_bounds
""")

import json as _json
bounds_row = spark.sql('SELECT * FROM cleaning_bounds').collect()[0].asDict()
BOUNDS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(BOUNDS_PATH, 'w') as fh:
    _json.dump({k: float(v) for k, v in bounds_row.items()}, fh, indent=2)
print(f'Bounds: {bounds_row}')
spark.sql('SELECT * FROM cleaning_bounds').show(truncate=False)


Bounds: {'fare_low': 6.0, 'fare_high': 232.10999999999999, 'speed_low': 2.337662337662338, 'speed_high': 47.23473541383989}


+--------+------------------+------------------+-----------------+
|fare_low|fare_high         |speed_low         |speed_high       |
+--------+------------------+------------------+-----------------+
|6.0     |232.10999999999999|2.3688311688311687|47.23473541383989|
+--------+------------------+------------------+-----------------+



In [4]:
# Auditoria de qualidade: contar linhas com cada problema antes do filtro
spark.sql("""
WITH base AS (
    SELECT *,
        trip_length / (total_ride_time / 3600.0) AS speed_mph_raw
    FROM trips_bronze
),
cb AS (SELECT * FROM cleaning_bounds)
SELECT
    COUNT(*) AS total_bronze,
    SUM(CASE WHEN passenger_fare <= 0 OR passenger_fare IS NULL THEN 1 ELSE 0 END) AS invalid_fare,
    SUM(CASE WHEN trip_length <= 0 THEN 1 ELSE 0 END) AS invalid_miles,
    SUM(CASE WHEN total_ride_time < 60 THEN 1 ELSE 0 END) AS too_short,
    SUM(CASE WHEN pickup_location IN (264, 265) OR dropoff_location IN (264, 265) THEN 1 ELSE 0 END) AS invalid_zones,
    SUM(CASE WHEN request_to_pickup < 0 THEN 1 ELSE 0 END) AS negative_wait
FROM base CROSS JOIN cb
""").show()


+------------+------------+-------------+---------+-------------+-------------+
|total_bronze|invalid_fare|invalid_miles|too_short|invalid_zones|negative_wait|
+------------+------------+-------------+---------+-------------+-------------+
|   116225448|       39628|        19596|    10925|      4806254|      1061782|
+------------+------------+-------------+---------+-------------+-------------+



In [5]:
# Silver: feature engineering + limpeza por percentil
# Excluidos: driver_total_pay (regulatorio), rideshare_profit (derivado do target),
#            hourly_rate e dollars_per_mile (LEAKAGE: computados a partir de passenger_fare),
#            on_scene_to_pickup / on_scene_to_dropoff (cobertura desigual entre operadoras)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW trips_silver AS
WITH base AS (
    SELECT
        b.*,
        trip_length / (total_ride_time / 3600.0) AS speed_mph,
        pu_zone.Borough AS pu_borough,
        do_zone.Borough AS do_borough
    FROM trips_bronze b
    LEFT JOIN taxi_zone_lookup pu_zone ON b.pickup_location  = pu_zone.LocationID
    LEFT JOIN taxi_zone_lookup do_zone ON b.dropoff_location = do_zone.LocationID
)
SELECT
    business,
    pickup_location,
    dropoff_location,
    trip_length,
    total_ride_time,
    request_to_pickup,
    passenger_fare,
    hour_of_day,
    month_of_year,
    week_of_year,
    time_of_day,
    date,
    pu_borough,
    do_borough,
    speed_mph,
    DAYOFWEEK(date)  AS pickup_dow,
    CASE WHEN DAYOFWEEK(date) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend,
    CASE WHEN hour_of_day BETWEEN 7 AND 9 OR hour_of_day BETWEEN 16 AND 19 THEN 1 ELSE 0 END AS is_rush_hour,
    CASE WHEN hour_of_day >= 22 OR hour_of_day <= 4 THEN 1 ELSE 0 END AS is_late_night,
    CASE WHEN pickup_location  IN (1, 132, 138) THEN 1 ELSE 0 END AS pickup_airport,
    CASE WHEN dropoff_location IN (1, 132, 138) THEN 1 ELSE 0 END AS dropoff_airport,
    -- NULL-safe: zona desconhecida -> 0
    CASE
        WHEN pu_borough IS NULL OR do_borough IS NULL THEN 0
        WHEN pu_borough = do_borough THEN 1
        ELSE 0
    END AS same_borough,
    DATE_FORMAT(date, 'yyyy-MM') AS pickup_year_month
FROM base
CROSS JOIN cleaning_bounds cb
WHERE passenger_fare BETWEEN cb.fare_low AND cb.fare_high   -- percentil 0.1% - 99.9%
  AND trip_length > 0                                        -- distancia positiva
  AND total_ride_time BETWEEN 60 AND 14400                   -- 1 min a 4 h
  AND request_to_pickup >= 0                                 -- espera nao negativa
  AND speed_mph BETWEEN cb.speed_low AND cb.speed_high      -- percentil velocidade
  AND speed_mph > 0.5                                        -- piso anti-ruido
  AND pickup_location  NOT IN (264, 265)                    -- Unknown / Outside NYC
  AND dropoff_location NOT IN (264, 265)
""")

spark.sql('SELECT COUNT(*) AS silver_rows FROM trips_silver').show()


+-----------+
|silver_rows|
+-----------+
|  110116813|
+-----------+



In [6]:
# DQ gate: mais de 20% de linhas removidas indica problema de calibracao
metrics_row = spark.sql("""
WITH total AS (SELECT COUNT(*) AS n FROM trips_bronze),
     silver AS (SELECT COUNT(*) AS n FROM trips_silver)
SELECT
    total.n AS bronze_count,
    silver.n AS silver_count,
    ROUND((total.n - silver.n) * 100.0 / total.n, 2) AS drop_pct
FROM total CROSS JOIN silver
""").collect()[0].asDict()
print(metrics_row)
assert metrics_row['drop_pct'] < 20, f"DQ gate: {metrics_row['drop_pct']}% removido (limite 20%)"
print('DQ gate OK')


{'bronze_count': 116225448, 'silver_count': 110116813, 'drop_pct': Decimal('5.26')}
DQ gate OK


In [7]:
# Materializa silver em Parquet particionado por mes para leitura eficiente nos modelos
(spark.table('trips_silver')
    .repartition('pickup_year_month')
    .write
    .partitionBy('pickup_year_month')
    .mode('overwrite')
    .parquet(SILVER_PATH))
print(f'Silver escrito em {SILVER_PATH}')


Silver escrito em /data/silver/trips_silver


In [8]:
spark.read.parquet(SILVER_PATH).createOrReplaceTempView('trips_silver_disk')
spark.sql("""
SELECT pickup_year_month, COUNT(*) AS trips
FROM trips_silver_disk
GROUP BY 1 ORDER BY 1
""").show(100, truncate=False)


+-----------------+--------+
|pickup_year_month|trips   |
+-----------------+--------+
|2023-03          |19345807|
|2023-04          |18159527|
|2023-05          |18777076|
|2023-06          |18336484|
|2023-07          |18140472|
|2023-08          |17357447|
+-----------------+--------+



In [9]:
spark.stop()
